In [ ]:
import os
from getpass import getpass
os.environ["HF_TOKEN"] = getpass("HF token: ")

In [ ]:
print(bool(os.environ.get("HF_TOKEN")))   # має бути True

In [ ]:

import torch; print(torch.cuda.is_available())

In [ ]:
!git clone https://github.com/Luzefik/storozha-anti-ambush.git

In [ ]:
!ls



In [ ]:
cd storozha-anti-ambush

In [ ]:
ls

In [ ]:
!git checkout feat/project-scaffold
!git log --oneline -1
!pip install -q -e . --no-deps


In [ ]:
!pip install ultralytics
!pip install -q kaggle
!kaggle competitions download -c predicting-assessment-scores-hw1-ucu-ai-2026 -p data
!unzip -q -o data/*.zip -d data
!ls data
!pip install -q pandas numpy seaborn matplotlib scikit-learn xgboost
!pip install seaborn


In [ ]:
!python scripts/download_data.py
!python scripts/prepare_data.py

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import (GridSearchCV, KFold, RandomizedSearchCV,
                                     cross_val_predict, cross_validate)
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor
from ultralytics import YOLO

In [ ]:
from ultralytics import YOLO


model = YOLO("yolo26n.pt")
model.train(data="data/processed/yolo/data.yaml",
            imgsz=640, epochs=100, patience=20, batch=16, seed=42)   # гіперпараметри
results = model.predict(frames)


In [ ]:
from ultralytics import RTDETR
model = RTDETR("rtdetr-l.pt")
results = model.train(
        data="data.yaml",
        epochs=100,
        imgsz=640,
        batch=16,
        device=0,
        workers=8
    )


In [ ]:
model = RTDETR("runs/detect/train/weights/best.pt")

# Запуск детекції на відео
results = model.predict(
    source="шлях/до/вашого/video.mp4",
    save=True,          # зберегти відео з рамками детекції
    conf=0.25           # поріг впевненості
)

In [ ]:

from zipfile import Path


yolo = Path("data/processed/yolo")
m = pd.read_csv("config/splits/video_split_manifest.csv")
pool = m[m["split"] != "test"]

K = 4
order = pd.concat([pool[pool.is_night], pool[~pool.is_night].sort_values("n_frames", ascending=False)])
folds = [[] for _ in range(K)]
for i, v in enumerate(order["video"]):
    folds[i % K].append(v)

def lines(videos):
    return [f"./images/{v}/{p.name}" for v in videos
            for p in sorted((yolo / "images" / v).glob("frame_*.png"))]

for k, val_videos in enumerate(folds):
    train_videos = [v for v in pool["video"] if v not in val_videos]
    (yolo / f"fold{k}_train.txt").write_text("\n".join(lines(train_videos)) + "\n")
    (yolo / f"fold{k}_val.txt").write_text("\n".join(lines(val_videos)) + "\n")
    (yolo / f"fold{k}.yaml").write_text(yaml.safe_dump(
        {"train": f"fold{k}_train.txt", "val": f"fold{k}_val.txt",
         "names": {0: "waiter_fpv_drone"}}, sort_keys=False))
    print(k, "val:", val_videos)

In [ ]:
results = []
for k in range(K):
    model = YOLO("yolo11s.pt")
    model.train(data=f"data/processed/yolo/fold{k}.yaml", imgsz=640, epochs=50,
                patience=15, batch=16, seed=42,
                project="/content/drive/MyDrive/storozha/cv", name=f"fold{k}")
    results.append(model.metrics.box.map50)          # mAP@0.5 на val цього фолду

import numpy as np
print(f"mAP50 = {np.mean(results):.3f} ± {np.std(results):.3f}", results)